# PREDIÇÃO DO BLOCO DE MARLIM
Para cada modelo em `BASE_PATH`, prediz os tiles `../Dataset/marlim/patch_*` (os `.dat` já
estão em [0,1] — **nenhum pré-processamento extra**) e salva as probabilidades (sigmoid,
float32) em `BASE_PATH/<modelo>/marlim/patch_<id>/masks/*.dat`.

**Janela de inferência.** Os tiles do Marlim são 128³, mas os modelos novos foram treinados
em blocos menores (`img_size` = 64³, 32³, 32×64×16 …). Jogar o tile inteiro numa rede
treinada em 32³ muda as estatísticas do GroupNorm e o contexto que ela viu no treino — a
predição degrada sem avisar. Por isso a inferência anda de janela em janela **no tamanho
exato do treino** (`SlidingWindow`, lido do `info.json` de cada modelo) e soma as janelas
com peso Hann, para a emenda não aparecer.

O arquivo de saída continua sendo um tile inteiro por `.dat` de entrada — o `Analysis.ipynb`
monta o bloco igualzinho, seja qual for a janela. Modelo treinado em 128³ = uma janela só =
exatamente o comportamento antigo.

In [1]:
import os, sys, glob, json, gc
import numpy as np
import torch
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

sys.path.append('../Model')                 # Network/, utils/ vivem em Model/
from Network.index import ModelNetwork

In [2]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

gc.collect()
print(torch.__version__)              # versão do PyTorch
print(torch.cuda.is_available())      # True se detectou a GPU
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))  # nome da GPU

2.7.1+cu118
True
Quadro P6000


In [3]:
BASE_PATH  = '../Model/Backup'    # pasta dos modelos: '../Model/Backup' ou '../Marcia'
MARLIM_DIR = '../Dataset/marlim'

PATCH_IDS = ['1200']
MODELS    = ['model_19']   # [] = todos os modelos de BASE_PATH
OVERLAP   = 0.25           # sobreposição entre janelas (0 = corte seco, do jeito que o Format corta o treino)
WINDOW    = None           # None = a janela do treino (info.json); force uma tupla para testar outra, ex: (64, 64, 64)
SKIP_DONE = False          # True pula o patch que já tem todas as máscaras

models = MODELS or sorted(os.listdir(BASE_PATH), key=lambda n: int(n.split('_')[-1]))
models

['model_19']

In [4]:
# ESCALAR OU LISTA VIRA TUPLA DE 3 EIXOS, PARA JANELA E TILE ENTRAREM NO MESMO FORMATO
def asTriple(value, default=None):
    value = default if value is None else value
    value = [value] * 3 if np.isscalar(value) else list(value)

    if len(value) != 3:
        raise ValueError(f'esperado 3 eixos, recebido {value}')

    return tuple(int(v) for v in value)


# LÊ O .DAT CRU DO MARLIM NO SHAPE DO PATCH (NÃO É FIXO EM 128³)
def getDAT(path, shape):
    data = np.fromfile(path, dtype=np.float32)

    if data.size != int(np.prod(shape)):
        raise ValueError(f'{os.path.basename(path)}: {data.size} valores, esperado {tuple(shape)}')

    return data.reshape(shape)


# SERVE OS TILES DO PATCH UM A UM, NO SHAPE DECLARADO NO PATCH_METADATA
class PredictDataset(Dataset):
    def __init__(self, patchDir, shape):
        self.paths = sorted(glob.glob(os.path.join(os.path.abspath(patchDir), '*.dat')))
        self.shape = asTriple(shape, default=128)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        return torch.from_numpy(getDAT(self.paths[index], self.shape))


PredictDataset(f'{MARLIM_DIR}/patch_{PATCH_IDS[0]}', 128).shape

(128, 128, 128)

# JANELA DE INFERÊNCIA
`SlidingWindow` prediz um tile inteiro andando na janela em que a rede foi treinada.

Cada janela sai da rede com o mesmo tamanho que ela viu no treino; as janelas vizinhas se
sobrepõem (`OVERLAP`) e são somadas com peso Hann — o centro da janela manda, a borda só
costura — de modo que a emenda não vira degrau na máscara. `OVERLAP = 0` reproduz o corte
seco que o `Format.ipynb` usa para tilar o treino; `janela == tile` cai numa passada só,
idêntica ao código antigo de 128³.

In [ ]:
# PREDIZ O TILE INTEIRO ANDANDO NA JANELA EM QUE A REDE FOI TREINADA
class SlidingWindow:
    BUDGET    = 128 ** 3   # voxels por lote na GPU
    MAX_BATCH = 256

    def __init__(self, window, shape, overlap=OVERLAP):
        self.tile    = asTriple(shape)
        self.window  = asTriple(window)
        self.overlap = float(overlap)

        self.shape  = tuple(max(t, w) for t, w in zip(self.tile, self.window))   # tile menor que a janela -> reflect
        self.pad    = tuple(s - t for s, t in zip(self.shape, self.tile))
        self.starts = [self.getStarts(s, w) for s, w in zip(self.shape, self.window)]

        self.positions = [(d, h, w) for d in self.starts[0] for h in self.starts[1] for w in self.starts[2]]
        self.weight    = self.getWeight(self.window)
        self.batch     = int(np.clip(self.BUDGET // int(np.prod(self.window)), 1, self.MAX_BATCH))

    # INÍCIO DE CADA JANELA NUM EIXO; A ÚLTIMA ENCOSTA NO FIM DO TILE
    def getStarts(self, size, window):
        step = max(int(round(window * (1 - self.overlap))), 1)
        return sorted(set(list(range(0, size - window + 1, step)) + [size - window]))

    # PESO 3D SEPARÁVEL, ~1 NO CENTRO DA JANELA E ~0 NA BORDA, PARA A EMENDA NÃO VIRAR DEGRAU
    @staticmethod
    def getWeight(window):
        d, h, w = [np.hanning(n + 2)[1:-1].astype(np.float32) for n in window]
        return np.maximum(d[:, None, None] * h[None, :, None] * w[None, None, :], 1e-3)

    # PROBABILIDADE DO TILE INTEIRO, SOMANDO AS JANELAS PONDERADAS PELO PESO
    def process(self, network, tile):
        volume = tile.numpy() if torch.is_tensor(tile) else np.asarray(tile)
        volume = np.pad(volume, [(0, p) for p in self.pad], mode='reflect') if any(self.pad) else volume
        volume = torch.from_numpy(np.ascontiguousarray(volume, np.float32)).to(network.device)

        weight = torch.from_numpy(self.weight).to(network.device)
        total  = torch.zeros((network.classes, *self.shape), dtype=torch.float32, device=network.device)
        norm   = torch.zeros(self.shape, dtype=torch.float32, device=network.device)
        wd, wh, ww = self.window

        with torch.no_grad():
            for start in range(0, len(self.positions), self.batch):
                chunk  = self.positions[start:start + self.batch]
                batch  = torch.stack([volume[d:d + wd, h:h + wh, w:w + ww] for d, h, w in chunk]).unsqueeze(1)
                logits = network.model(batch)
                probs  = torch.softmax(logits, dim=1) if network.multiclass else torch.sigmoid(logits)

                for i, (d, h, w) in enumerate(chunk):
                    total[:, d:d + wd, h:h + wh, w:w + ww] += probs[i].float() * weight
                    norm[d:d + wd, h:h + wh, w:w + ww]     += weight

        probs = (total / norm).cpu().numpy().astype(np.float32)
        probs = probs[0] if network.classes == 1 else probs
        return probs[..., :self.tile[0], :self.tile[1], :self.tile[2]]

    def __repr__(self):
        return (f'SlidingWindow(janela={self.window}, tile={self.tile}, overlap={self.overlap}, 'f'janelas/tile={len(self.positions)}, batch={self.batch})')


SlidingWindow(window=(32, 32, 32), shape=128)

SlidingWindow(janela=(32, 32, 32), tile=(128, 128, 128), overlap=0.25, janelas/tile=125, batch=64)

In [6]:
# PREDIZ OS PATCHES DO MARLIM COM CADA MODELO, NA JANELA EM QUE ELE FOI TREINADO
class MarlimPredictor:
    def __init__(self, models, base=BASE_PATH, marlimDir=MARLIM_DIR, patchIds=PATCH_IDS,
                 overlap=OVERLAP, window=WINDOW, skipDone=SKIP_DONE):
        self.models    = models
        self.base      = base
        self.marlimDir = marlimDir
        self.patchIds  = patchIds
        self.overlap   = overlap
        self.window    = window
        self.skipDone  = skipDone

    # PASTA DAS MÁSCARAS DE UM PATCH
    def outputDir(self, modelName, pid):
        return os.path.join(self.base, modelName, 'marlim', f'patch_{pid}', 'masks')

    # CARREGA OS PESOS E DEVOLVE A REDE COM A JANELA EM QUE ELA FOI TREINADA
    def loadNetwork(self, modelName):
        modelPath = f'{self.base}/{modelName}'
        options   = json.load(open(f'{modelPath}/info.json', encoding='utf-8')).get('model', {})

        print(f'[{modelName}] Model Options:')
        print(json.dumps(options, indent=4))

        network = ModelNetwork(**options)
        network.model.load_state_dict(torch.load(f'{modelPath}/model.pth', map_location=network.device)['model'])
        network.model.eval()
        print(f'Weights from {modelPath} loaded successfully!\n')
        return network, asTriple(self.window or options.get('img_size'), default=128)

    # LOADER QUE SERVE OS TILES DO PATCH, NO SHAPE DECLARADO NO PATCH_METADATA
    def getLoader(self, pid):
        patchDir = f'{self.marlimDir}/patch_{pid}'
        metaPath = f'{patchDir}/patch_metadata.json'
        meta     = json.load(open(metaPath)) if os.path.exists(metaPath) else {}
        dataset  = PredictDataset(patchDir, meta.get('patch_size'))

        if not len(dataset):
            raise FileNotFoundError(f'nenhum .dat em {patchDir}')

        return DataLoader(dataset, batch_size=1, shuffle=False, num_workers=2,
                          pin_memory=torch.cuda.is_available())

    # TRUE QUANDO A masks/ JÁ TEM UM TILE PARA CADA .DAT DE ENTRADA
    def isDone(self, modelName, pid):
        done  = glob.glob(os.path.join(self.outputDir(modelName, pid), '*.dat'))
        total = glob.glob(f'{self.marlimDir}/patch_{pid}/*.dat')
        return len(total) > 0 and len(done) >= len(total)

    # REGISTRA A JANELA USADA NO PATCH, QUE O ANALYSIS LÊ PARA A TABELA
    def export(self, modelName, pid, sliding, tiles):
        info = {
            'model': modelName,
            'patch': pid,
            'tiles': tiles,
            'tile_shape': list(sliding.tile),
            'img_size': list(sliding.window),
            'overlap': sliding.overlap,
            'windows_per_tile': len(sliding.positions),
        }

        path = os.path.join(self.base, modelName, 'marlim', f'patch_{pid}', 'predict.json')
        json.dump(info, open(path, 'w', encoding='utf-8'), ensure_ascii=False, indent=4)

    # PREDIZ TODOS OS TILES DO PATCH, GRAVANDO UMA MÁSCARA POR TILE NO MESMO SHAPE DA ENTRADA
    def predictPatch(self, network, window, modelName, pid):
        loader    = self.getLoader(pid)
        dataset   = loader.dataset
        sliding   = SlidingWindow(window, dataset.shape, overlap=self.overlap)
        outputDir = self.outputDir(modelName, pid)
        os.makedirs(outputDir, exist_ok=True)

        print(f'[{modelName}] patch_{pid}: {sliding}')

        for index, tile in enumerate(tqdm(loader, desc=f'{modelName} - patch_{pid}')):
            sliding.process(network, tile[0]).tofile(os.path.join(outputDir, os.path.basename(dataset.paths[index])))

        self.export(modelName, pid, sliding, len(dataset))
        print(f'Predições de {modelName} - patch_{pid} salvas em {outputDir}')

    # RODA CADA MODELO DA LISTA EM TODOS OS PATCHES PENDENTES
    def start(self):
        for modelName in self.models:
            todo = [pid for pid in self.patchIds if not (self.skipDone and self.isDone(modelName, pid))]

            if not todo:
                print(f'[{modelName}] patches já preditos — pulando')
                continue

            if not os.path.exists(f'{self.base}/{modelName}/model.pth'):
                print(f'[{modelName}] sem model.pth — pulando')
                continue

            network, window = self.loadNetwork(modelName)

            for pid in todo:
                self.predictPatch(network, window, modelName, pid)

            del network
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


predictor = MarlimPredictor(models)
predictor.start()

[model_19] Model Options:
{
    "network": "resaceunet_grva",
    "img_size": [
        128,
        128,
        128
    ],
    "classes": 1,
    "channels": 1,
    "dropout": 0.1,
    "num_filters": 32,
    "lr": 0.001
}
Weights from ../Model/Backup/model_19 loaded successfully!

[model_19] patch_1200: SlidingWindow(janela=(128, 128, 128), tile=(128, 128, 128), overlap=0.25, janelas/tile=1, batch=1)


model_19 - patch_1200: 100%|██████████| 910/910 [06:34<00:00,  2.31it/s]


Predições de model_19 - patch_1200 salvas em ../Model/Backup/model_19/marlim/patch_1200/masks
